# Module 6 — Chemical equilibrium, two ways

**2105603 Advanced Chemical Engineering Thermodynamics**
Department of Chemical Engineering, Chulalongkorn University
Soorathep Kheawhom

---

Two parts, and the first one is a cross-check rather than a calculation.

| part | what you do |
|---|---|
| 1 | solve one equilibrium **twice** — by extent and by constrained Gibbs minimisation — and require the two answers to agree |
| 2 | a Pitzer calculation of mean ionic activity, and a cell voltage against concentration compared with measured data |

**Why part 1 is built this way.** The Gibbs objective contains
$\sum_i n_i \ln(n_i/n)$, which is ill-conditioned as any mole number approaches
zero. Your code *will* fail on it. The extent formulation is exact wherever a
reaction set is known, so it is the instrument that detects the failure. A
converged optimisation is not an answer.

## 0. Setup

In [ ]:
import sys, subprocess, importlib, os

def ensure(pkg, pipname=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        pipname or pkg], check=False)

for p in ("numpy", "scipy", "matplotlib"):
    ensure(p)
ensure("chemicals")        # thermochemical data; required for this module

if not os.path.isdir("vlekit"):
    import urllib.request, zipfile, io
    URL = "https://www.skhgroup.net/teaching/2105603/files/vlekit.zip"
    try:
        with urllib.request.urlopen(URL, timeout=30) as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(".")
        print("downloaded vlekit from the course page")
    except Exception as e:
        print("could not download vlekit:", e)

import numpy as np
import matplotlib.pyplot as plt
import vlekit as v
from vlekit import reaction as X, plots as pl

pl.use_style()
print("vlekit", v.__version__)

## 1. The data, and why they are not typed in

Standard formation properties copied by hand are wrong more often than any
other quantity in this subject, and an error of 1 kJ/mol in $\Delta G^\circ$ is
a **50 % error in $K$** at 298 K. So the data come from the `chemicals`
package, and the first thing to do is check them against something else.

In [ ]:
names = ["nitrogen", "hydrogen", "ammonia", "carbon monoxide",
         "carbon dioxide", "water", "methane"]
sp = {n: X.from_chemicals(n) for n in names}

lit = {"ammonia": (-45.9, 192.8), "carbon monoxide": (-110.5, 197.7),
       "carbon dioxide": (-393.5, 213.8), "water": (-241.8, 188.8),
       "methane": (-74.6, 186.3)}
print(f"{'species':18s}{'Hf/kJ':>9}{'lit':>9}{'S0':>9}{'lit':>9}{'Cp(298)':>10}")
for n, (h, s) in lit.items():
    c = sp[n]
    print(f"{n:18s}{c.Hf/1000:>9.1f}{h:>9.1f}{c.S0:>9.1f}{s:>9.1f}{c.Cp(298.15):>10.1f}")

In [ ]:
# The cleanest published check: water-gas shift.
wgs = X.reaction({sp["carbon monoxide"]: -1, sp["water"]: -1,
                  sp["carbon dioxide"]: 1, sp["hydrogen"]: 1}, None)
print(wgs, " element balance:", wgs.element_balance())
print(f"dH(298) = {wgs.dH(298.15)/1000:7.2f} kJ/mol   literature -41.2")
print(f"K(298)  = {wgs.K(298.15):10.3e}          literature  1.0e5")
print(f"K(1000) = {wgs.K(1000.0):10.3f}          literature  ~1.4")

nh3 = X.reaction({sp["nitrogen"]: -1, sp["hydrogen"]: -3, sp["ammonia"]: 2},
                 None, "ammonia synthesis")
print(f"\n{nh3}")
print(f"dG(298) = {nh3.dG(298.15)/1000:7.2f} kJ/mol   literature -32.8")
print(f"K(298)  = {nh3.K(298.15):10.3e}          literature  5.9e5")
print(f"\nthat 0.75 kJ/mol difference in dG makes K "
      f"{1 - nh3.K(298.15)/5.9e5:.0%} smaller than the literature value.")
print("1 kJ/mol in dG is a factor", round(float(np.exp(1000/(8.314462618*298.15))),3),
      "in K at 298 K. Report your data source.")

**Question 1.1.** Recompute $K$ for the shift reaction with $\Delta H^\circ$
held constant at its 298 K value (`K_vant_hoff`). At what temperature do the
two differ by 10 %? Would you have guessed that from the size of
$\Delta C_p$?

## 2. The same equilibrium, solved twice

The extent formulation is told the stoichiometry and solves one scalar
equation. The Gibbs minimisation is told only the element list and never learns
that a reaction exists. **They must agree.**

In [ ]:
n0 = np.array([1.0, 3.0, 0.0])                # N2 : H2 : NH3
species = [sp["nitrogen"], sp["hydrogen"], sp["ammonia"]]

print(f"{'T/K':>7}{'P/bar':>8}{'xi':>11}{'y(NH3) extent':>15}"
      f"{'y(NH3) Gibbs':>14}{'|diff|':>10}")
worst = 0.0
for T in (600.0, 700.0):
    for Pbar in (1.0, 50.0, 200.0, 300.0):
        P = Pbar * 100.0                       # kPa
        e = X.equilibrium_composition(nh3, n0, T, P)
        g = X.gibbs_minimise(species, n0, T, P, restarts=8)
        d = abs(e["y"][2] - g["y"][2]); worst = max(worst, d)
        print(f"{T:>7.0f}{Pbar:>8.0f}{e['xi']:>11.6f}{e['y'][2]:>15.6f}"
              f"{g['y'][2]:>14.6f}{d:>10.1e}")
print(f"\nworst disagreement over the whole table: {worst:.2e} in mole fraction")
assert worst < 1e-4, "the two formulations disagree — do not trust either"

**Question 2.1.** Set `restarts=1` and re-run. Does anything change? Now start
the minimisation from a composition that is nearly pure ammonia. What does that
tell you about the objective?

**Question 2.2.** The extent solver uses bisection, not Newton. Read
`solve_extent` and say why bisection cannot fail here.

In [ ]:
# The element potentials, and the relation that checks the solution for free.
g = X.gibbs_minimise(species, n0, 700.0, 30000.0, restarts=8)
lam = dict(zip(g["elements"], g["element_potentials"] / 1000))
print("element potentials, kJ per mol of atom:",
      {k: round(float(x), 2) for k, x in lam.items()})

mu = X.chemical_potentials(species, g["n"], 700.0, 30000.0)
A, els = X.element_matrix(species)
print("\nspecies        mu/kJ    sum_k a_ki lambda_k")
for i, s in enumerate(species):
    print(f"{s.name:14s}{mu[i]/1000:8.2f}{float(A[:, i] @ g['element_potentials'])/1000:>22.2f}")
print(f"\nmax residual: "
      f"{X.element_potential_residual(species, g['n'], 700.0, 30000.0):.3e} J/mol")
print("A species carries no equilibrium information beyond the atoms it is "
      "made of.")

## 3. Where the extent formulation cannot help

Steam reforming: six species, three elements, and a reaction set you have to
choose. Solve it by Gibbs minimisation, and cross-check with two independent
reactions where you can.

In [ ]:
smr = [sp["methane"], sp["water"], sp["carbon monoxide"],
       sp["hydrogen"], sp["carbon dioxide"]]
n0s = np.array([1.0, 3.0, 0.0, 0.0, 0.0])          # steam-to-carbon = 3
Ts = np.arange(600.0, 1301.0, 50.0)

Y = []
for T in Ts:
    gg = X.gibbs_minimise(smr, n0s, T, 100.0, restarts=6)
    Y.append(gg["y"])
Y = np.array(Y)

fig, ax = pl.newfig(7.0, 4.4)
for i, s in enumerate(smr):
    ax.plot(Ts, Y[:, i], "-", lw=2.0, label=s.name)
ax.set_xlabel("$T$ / K"); ax.set_ylabel("mole fraction")
ax.legend(loc="best", fontsize=9.5)
ax.set_title("Steam reforming, S/C = 3, 1 bar — from Gibbs minimisation")
plt.show()

**Question 3.1.** Read the chemistry off the curves. Where does methane
disappear? Which way does the shift equilibrium move as $T$ rises, and why?

**Question 3.2.** Add solid carbon to the species list and re-run at
steam-to-carbon of 1. Does the minimiser find it? Compare with what the carbon
element potential says on its own. Coking is on the thermodynamics, not only in
the kinetics.

## 4. Electrolytes and a measurable voltage

Three models of $\gamma_\pm$, one measurement. The Pitzer parameters below are
the Pitzer & Mayorga set for HCl distributed with PHREEQC; $E^\circ$ and the
cell data are from Bates & Bower (1954).

In [ ]:
m = np.array([0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1])
I = m.copy()                                      # 1:1 electrolyte

g_lim = X.dh_limiting(I, 1, -1)
g_ext = X.dh_extended(I, 1, -1, a_ang=4.3)
g_pit = X.pitzer(I, m, 1, -1, beta0=0.1775, beta1=0.2945, Cphi=0.0008)

print(f"A(298.15) = {X.debye_huckel_A(298.15):.4f}  (literature 0.5108)")
print(f"\n{'m':>8}{'limiting':>11}{'extended':>11}{'Pitzer':>10}")
for i, mi in enumerate(m):
    print(f"{mi:>8.3f}{g_lim[i]:>11.4f}{g_ext[i]:>11.4f}{float(g_pit[i]):>10.4f}")

In [ ]:
E0 = 0.22234                                    # V, Ag/AgCl
T = 298.15
E_ideal = X.cell_voltage(E0, T, 1, m, 1.0, nu_ions=2)
E_pit   = X.cell_voltage(E0, T, 1, m, np.asarray(g_pit, float), nu_ions=2)

print(f"{'m':>8}{'E ideal/V':>12}{'E Pitzer/V':>12}{'difference/mV':>15}")
for i, mi in enumerate(m):
    print(f"{mi:>8.3f}{E_ideal[i]:>12.5f}{E_pit[i]:>12.5f}"
          f"{1000*(E_pit[i]-E_ideal[i]):>15.2f}")
print("\nAt 0.1 molal the activity coefficient is worth over ten millivolts,")
print("against a measurement uncertainty of about 0.01 mV.")

**Question 4.1.** The Nernst slope is $2.303\,RT/F = 59.16$ mV per decade at
25 °C. Verify it from `nernst` before trusting anything above, and say what a
wrong electron count would do to it.

**Question 4.2.** Which model would you use for a battery electrolyte at
1 mol/kg? At 5 mol/kg? Justify with the numbers, not with the name.

---

## Exercises

**E1. Your own reaction.** Choose an industrially relevant equilibrium, solve
it both ways, and report the disagreement. If the two disagree by more than
$10^{-4}$ in mole fraction, find out why before reporting anything else.

**E2. The correction hierarchy.** For a high-pressure gas equilibrium, compute
the effect on conversion of (a) fugacity coefficients, (b) a 1 kJ/mol
uncertainty in $\Delta G^\circ$, (c) a $\pm 0.05$ change in every $k_{ij}$.
Rank them, and say where you would spend effort.

**E3. Break the minimiser.** Add a species whose equilibrium mole fraction is
below $10^{-10}$ and watch the element-potential residual. Report the mole
fraction at which the method stops being trustworthy on your system.

**E4. Standard states.** Take one reaction in solution and compute $K$ on the
mole fraction, molality and molarity scales. Show that the equilibrium
composition is the same in all three.

**E5. Carbon.** For steam reforming, map the region in (temperature,
steam-to-carbon) where solid carbon is stable, using the carbon element
potential. Compare with the rule of thumb your reforming textbook gives.

---

## AI checkpoint

Ask a language model to set up the Gibbs minimisation for a feed of your
choosing — species list, element matrix, constraint vector, objective.

Then check it. **Constraint errors are easy to make and easy to check**: apply
the element matrix to the feed and to a solution and see whether the atoms
balance. Report what the model got right, what it got wrong, and — the part
that matters — whether you could tell from the answer alone or had to run the
check.

A model that produces a plausible element matrix with one wrong coefficient
gives an equilibrium that converges, satisfies its own constraints, and is
wrong. That is the same failure mode as everything else in this course.

---

*vlekit and this notebook: Soorathep Kheawhom, 2105603, Chulalongkorn
University.*